PINNs - Redes Neurais Informadas por Física
====================

# Teoria 

## O que são?

Uma __Physics-Informed Neural Network__ (_PINN_) ou, em português claro, _Rede Neural Informada por Física_ é uma abordagem de aprendizado profundo em que uma rede neural é treinada para aproximar a solução de uma equação diferencial ao inserir leis físicas diretamente na função de perda. 

A proposta primária é justamente incorporar as limitações físicas fundamentais, normalmente na forma de equações diferenciais parciais não lineares (EDPs não lineares), ao invés de basear-se apenas em um grande volume de dados para guiar o processo de aprendizagem. 

Diferentemente de abordagens tradicionais de aprendizado de máquina, que dependem fortemente de grandes volumes de dados, as _PINNs_ exploram conhecimento a priori sobre o sistema, permitindo aprender soluções fisicamente consistentes mesmo em regimes com poucos dados disponíveis.

## Breve Histórico

### Antecedentes

A ideia de utilizar redes neurais para resolver equações diferenciais antecede o acrônimo _PINN_ por praticamente duas décadas. O trabalho considerado fundamental é de Lagaris, Likas e Fotiadis (1998) [1], que propuseram escrever _trial solutions_ como somas de duas partes - uma satisfazendo as condições de contorno exatamente e uma segunda envolvendo uma rede de feedfoward (aquelas em que os dados fluem em uma única direção sem ciclos ou feedback). 

Para essa aplicação, é interessante entender o que são as _trial solutions_. A tradução mais comum é **função tentativa**, uma função construída artificialmente para garantir que ela satisfaça automaticamente as condições de contorno e que dependa de uma função ajustável a ser otimizada.

Eles demonstraram essa abordagem em EDOs e EDPs, comparando-se favoravelmente contra outros métodos computacionais, apesar de ser consideravelmente anterior à era moderna do aprendizado profundo. 

### A introdução do framework

Os responsáveis pela construção moderna das _PINNs_ foram Maziar Raissi, Paris Perdikaris e George Kaniadakis. Em artigo publicado em 2019 no *Journal of Computational Physics*[2], a referência acadêmica canônica sobre a matéria, eles fizeram uma contribuição chave; transformaram o problema em um framework de aprendizado multi-tarefa. Agora, a rede neural deveria simultaneamente conferir o ajuste dos dados observáveis **e** minimizar os resíduos das EDPs que modelam o sistema, avaliando as funções em pontos pelo domínio.

O grande salto foi fazer uso da diferenciação automática, presente e disponível em bibliotecas de aprendizado profundo como _TensorFlow_ ou _PyTorch_, para calcular derivadas da saída da rede em relação às entradas. Essa inovação forneceu uma vantagem prática com relação aos algoritmos anteriores que seguiam o estilo de Lagaris. Além disso, algo que distinguia ainda mais as _PINNs_ desses algoritmos era sua capacidade de não apenas resolver problemas diretamente, mas inversamente e também, através de descoberta guiada por dados (_data-driven discovery_) das equações que governam os sistemas estudados. 

## Conceitos Centrais

### A função de perda

Após entendermos um pouco a respeito das definições e da história das _PINNs_, devemos esmiuçar os seus meandros para uma compreensão completa. Como foi supracitado, há uma diferença fundamental que separa essas redes de redes neurais tradicionais: a incorporação da física. 

Mas como isso é de fato incorporado do ponto de vista matemático? O que é esse objeto novo que difere tanto as _PINNs_ de outras redes?

Uma rede neural tradiconal tem como objetivo minimizar a diferença do resultado previsto pela rede e o resultado real (a **perda dos dados**). Estrutura-se uma **função de perda** e o objetivo da rede é, tradicionalmente utilizando os algoritmos de retropropagação e descida do gradiente, minimizá-la. Já em uma _PINN_ devemos introduzir um segundo componente para o objetivo de treinamento: a **perda física** ou a _perda residual_. Dessa forma, reestrutura-se a função de perda; a rede é treinada para minimizar uma função de perda de múltiplos objetivos:

$L_{total} = w_{data}L_{data} \; + \; w_{fisico}L_{fisico} $

Onde:

- $L_{data} \;$ mede a discrepância entre a saída da rede e dados observáveis disponíveis;
- $L_{fisico} \;$  penaliza o modelo se as predições violarem as equações físicas que regem o sistema. 

Os $w$'s são os pesos de cada uma das perdas. 

É importante notar que, em muitos problemas, as condições iniciais e de contorno também podem ser incorporadas como termos adicionais na função de perda, desempenhando um papel análogo ao de dados observados.

Nas seções seguintes, refinaremos essa formulação, escrevendo explicitamente a função de perda em termos dos resíduos da equação diferencial e das restrições impostas pelas condições do problema.

### Resíduos na perda física

#### O que é o resíduo?

Precisamos entender, porém, como é construído $L_{fisico}$ para, então, pensar em métodos de otimização que façam sentido para _PINNs_. Vamos começar por entender o que são **resíduos** no contexto das EDPs. Consideremos uma EDP geral definida pelo operador não-linear $\mathcal{F}$:

$$
\mathcal{F}[u(x,t)] = f (x,t)
$$

Onde $u(x,t)$ é a solução exata para a EDP. Se houve uma solução aproximada, denotada de $\hat{u}(x,t)$, o **resíduo** $R(x,t)$ é definido como a diferença entre o operador aplicado à aproximação e o termo original $f$:

$$
R(x,t) = \mathcal{F}[\hat{u}(x,t)] - f(x,t)
$$

Note que se $\hat{u}$ for a solução exata, o resíduo torna-se 0 em todos os pontos. Em métodos numéricos como o _PINNs_, a ideia é minimizar o resíduo ao longo do domínio. 
Além desse mecanismo clássico, _PINNs_ podem utilizar o resíduo para descoberta de parâmetros, como uma constante física desconhecida $\lambda$. Dessa forma, modifca-se sua fórmula para que $\lambda$ seja incorporado:

$$
R(x, t; \lambda) = \mathcal{F}[\hat{u}(x,t);\lambda] - f(x,t)
$$

É importante fazer uma diferenciação conceitual também. **O resíduo é diferente do erro**. Em uma tarefa comum de aprendizado supervisionado, mede-se o erro por comparação entre a predição do modelo e o dado real. Nesse framework, se a solução exata é desconhecida, não conseguimos calcular o erro e obter métricas para melhorar o modelo. _PINNs_ **não operam com erro para a função de perda, mas sim com resíduo**. O princípio por trás é a **consistência** ao invés da comparação: o resíduo mede o quão bem uma aproximação satisfaz uma lógica interna das equações diferenciais, independentemente se uma solução fechada para o problema existe. 

#### Como ele aparece em _PINNs_?

Em _PINNs_ a própria rede funciona como a função aproximada $\hat{u}(x,t; \theta)$ onde $\theta$ representa os parâmetros de pesos e vieses. O resíduo é utilizado então para construir o termo de perda física, que informa a rede sobre as limitações físicas do sistema.

Olhemos para o mecanismo passo a passo:

- Pontos de colocação: uma amostra contendo um conjunto de pontos é selecionada no domínio. Esses pontos são chamados de pontos de colocação (_collocation points_) porque não precisamos dos valores verdadeiros de $u$ para eles, somente como a física se aplica nesses pontos;
- Diferenciação automática (DA): para calcular o resíduo, a rede computa as derivadas com respeito aos dados de entrada. Ao contrário métodos numéricos tradicionais que utilizam métodos baseados em grid (discretizam o espaço de dados em um número finito de células para formar uma estrutura), _PINNs_ usam DA para computar as derivadas nos pontos de colocação;
- Construindo a perda: o resíduo é calculado para todos esses pontos e a função de perda física é, tipicamente, o erro quadrático médio (MSE) dos resíduos:
            $$L_{fisico} = \dfrac{1}{N_f}\sum^{N_f}_{i=1} |R(x_i, t_i)|^{2} $$


    Assim, a perda física pode ser interpretada como a minimização do erro do operador diferencial no domínio;

- Otimização: o otimizador de escolha ajusta $\theta$ para minimizar a perda física. Quando $L_{fisico} \rightarrow 0$, a rede neural é forçada a satisfazer a EDP.

Note que a equação diferencial é satisfeita apenas nos pontos amostrados, sendo a generalização para o restante do domínio dependente da capacidade de aproximação da rede.

#### Retornando à função de perda

Agora, faz mais sentido escrever explicitamente a função de perda e interpretar seus termos. A função de perda total de uma _PINN_ pode ser escrita como:

$$
\mathcal{L}(\theta) =
\underbrace{\frac{1}{N_r} \sum_{i=1}^{N_r} \left|\mathcal{N}[u_\theta(x_i,t_i)]\right|^2}_{\text{resíduo no domínio}}
+
\underbrace{\frac{1}{N_b} \sum_{j=1}^{N_b} \left|u_\theta(x_j) - g(x_j)\right|^2}_{\text{condições de contorno}}
+
\underbrace{\frac{1}{N_0} \sum_{k=1}^{N_0} \left|u_\theta(x_k, 0) - u_0(x_k)\right|^2}_{\text{condições iniciais}}
$$

onde cada termo desempenha um papel distinto no processo de treinamento da rede neural.

- O resíduo $R(x,t)$, definido como

$$
R(x,t) = \mathcal{N}[u_\theta](x,t),
$$

mede o quanto a função aproximada pela rede neural viola a equação diferencial no interior do domínio. O primeiro termo da função de perda corresponde ao erro quadrático médio desse resíduo, avaliado em pontos amostrados no domínio (_collocation points_).

- O segundo termo da função de perda está associado às condições de contorno. Nele, $g(x)$ representa a função que define os valores que a solução deve assumir nas fronteiras do domínio. Esse termo penaliza desvios entre a predição da rede $u_\theta(x)$ e os valores impostos no contorno, sendo avaliado em um conjunto de pontos de contorno $\{x_j\}_{j=1}^{N_b}$.

- O terceiro termo corresponde às condições iniciais. Considerando uma condição do tipo $u(x,0) = u_0(x)$, esse termo mede o erro quadrático médio entre a solução aproximada pela rede e os valores iniciais prescritos, avaliados em pontos $\{x_k\}_{k=1}^{N_0}$ no instante inicial.

Assim, o treinamento de uma _PINN_ pode ser interpretado como a busca por uma função $u_\theta(x,t)$ que simultaneamente:

- satisfaz a equação diferencial no domínio,
- respeita as condições de contorno,
- e atende às condições iniciais.

Na próxima seção, discutiremos com mais detalhe a natureza dessas condições e como elas atuam na restrição do espaço de soluções admissíveis.


#### Condições Iniciais e Condições de Contorno

Como, ao utilizar _PINNs_, estamos essencialmente navegando no espaço das soluções de equações diferenciais, precisamos de uma forma de restringir esse espaço a soluções fisicamente relevantes. Em geral, EDPs possuem infinitas soluções possíveis, a menos que sejam impostas condições adicionais que limitem esse conjunto. Essas restrições são conhecidas como **condições iniciais** e **condições de contorno**.

Em métodos numéricos tradicionais, como os métodos de elementos finitos, essas condições são tratadas como _hard constraints_, ou seja, são incorporadas diretamente na construção da solução de modo que não possam ser violadas.

Em _PINNs_, por outro lado, essas condições são tratadas como _soft constraints_. Isso significa que não são impostas de forma exata, mas sim incorporadas à função de perda como termos de penalização. Durante o treinamento, a rede neural é incentivada a satisfazer essas condições ao minimizar essas penalidades.

As **condições iniciais** definem o estado do sistema em um instante inicial, usualmente $t=0$. Matematicamente, são expressas como:

$$
u(x,0) = u_0(x)
$$

Em _PINNs_, essa condição é imposta selecionando um conjunto de pontos $\{x_k\}_{k=1}^{N_0}$ no domínio espacial e penalizando desvios da rede nesses pontos. Isso leva ao termo de perda:

$$
\frac{1}{N_0} \sum_{k=1}^{N_0} \left|u_\theta(x_k,0) - u_0(x_k)\right|^2
$$

De forma análoga, as **condições de contorno** especificam o comportamento da solução nas fronteiras do domínio espacial. Por exemplo, uma condição de Dirichlet pode ser escrita como:

$$
u(x_b,t) = g(x_b,t)
$$

para pontos $x_b$ localizados no contorno do domínio.

Em _PINNs_, essas condições são impostas selecionando pontos no contorno e penalizando a diferença entre a predição da rede e os valores prescritos:

$$
\frac{1}{N_b} \sum_{j=1}^{N_b} \left|u_\theta(x_j,t_j) - g(x_j,t_j)\right|^2
$$

Dessa forma, tanto as condições iniciais quanto as de contorno entram na função de perda como termos adicionais que guiam o processo de otimização.

O papel dessas condições é restringir o espaço de funções admissíveis, garantindo que a solução encontrada pela rede neural não apenas satisfaça a equação diferencial no domínio, mas também seja consistente com o comportamento físico esperado nas fronteiras e no instante inicial.

Dessa forma, o problema de resolver uma equação diferencial com uma _PINN_ é reformulado como um problema de otimização: busca-se encontrar os parâmetros $\theta$ que minimizam simultaneamente o resíduo da equação diferencial e as violações das condições iniciais e de contorno.

Entretanto, essa função de perda possui uma estrutura significativamente mais complexa do que aquelas encontradas em tarefas tradicionais de aprendizado de máquina. A presença de operadores diferenciais — frequentemente envolvendo derivadas de ordem superior — torna o espaço de otimização mais desafiador, com regiões de alta curvatura e múltiplos mínimos locais.

Isso levanta uma questão central: como otimizar eficientemente essa função de perda? Na próxima seção, discutiremos os métodos de otimização mais utilizados em _PINNs_ e as dificuldades específicas associadas a esse processo.

### Otimização

O conceito que faz com que as _PINNs_ sejam úteis são os métodos de diferenciação automática, como a retropropagação, para computar as derivadas. 

Aqui, porém, temos uma questão mais complicada de **otimização**, ou como a rede neural detecta a direção do ajuste e como fazê-lo. Redes neurais comuns quase exclusivamente fazem uso de métodos de primeira ordem, como Adam (_Adaptive Moment Estimation_), SGD (_Stochastic Gradient Descent_) ou Nesterov para realizar a otimização. Eles tem algumas vantagens:

- Rápidos
- Baixo custo de memória
- Escalonáveis para muitos parâmetros

Mas para _PINNs_, uma característica faz com que esses otimizadores não sejam os mais apropriados. O termo físico que inserimos na função de perda possui derivadas de ordem superior (algo do tipo $\dfrac{\partial ² u}{ \partial x²}$ ou superior), o que introduz estruturas de alta curvatura que podem dificultar a otimização para otimizadores de primeira ordem. Eles podem ficar presos em alguma condição ou demorar muito para convergir em uma resposta. Por isso, é comum que se utilizem otimizadores de segunda ordem ou, pelo menos, algo próximo. Esses algoritmos utilizam ambos gradiente e  Hessiano ($H$), ou a matriz de segundas derivadas. Essa utilização faz com que os algoritmos entendam não só qual a direção do ponto de mínimo, mas como a curvatura está se comportando. 

O método de segunda ordem ideal é o método de Newton, que realiza um salto em direção ao mínimo através da inversão do Hessiano. O impedimento é que, para $N$ parâmetros, o Hessiano é uma matriz $N \times N$. A complexidade de um código que realize essa inversão é cúbica, o que se torna imprático, até para redes de médio porte. É por isso que foi preciso buscar uma alternativa. 

Em _PINNs_, é comum que se utilizem métodos denominados Quasi-Newtonianos, como **BFGS** (Broyden–Fletcher–Goldfarb–Shanno) ou **L-BFGS** (Limited-memory BFGS). Esses métodos aproximam o inverso do Hessiano sem calculá-lo de maneira explícita.  

O método BFGS atualiza uma estimativa do inverso do Hessiano utilizando a **Equação da secante**:
$$B_{k+1} = (I - \rho_k s_k y_k^\top) B_k (I - \rho_k y_k s_k^\top) + \rho_k s_k s_k^\top$$

Onde 

* **$s_k = \theta_{k+1} - \theta_k$**: É a mudança nos parâmetros do modelo (pesos/vieses);
* **$y_k = \nabla L(\theta_{k+1}) - \nabla L(\theta_k)$**: É a mudança no gradiente da função de perda;
* **$\rho_k = \frac{1}{y_k^\top s_k}$**: É um fator escalar de curvatura que garante que a atualização permaneça positiva e definida;
* **$I$**: É a matriz identidade.

A versão L-BFGS limita o uso de memória por não armazenar a matriz $B_k$. Ao invés disso, armazena-se as últimas $m$ versões dos vetores $s_k \text{ e } y_k$. Após isso, utilizam-se dois laços recursivos para computar a direção de ajuste diretamente (LIU, NOCEDAL)[3]. 

De maneira prática, podemos comparar os métodos por convergência. Enquanto métodos lineares (como o Adam) convergem mais rápido, a sua precisão é menor do que um BFGS que converge de forma superlinear. Isso significa que, uma vez perto da solução, ele chega a ela com uma precisão muito maior. 

Além disso, vale notar que o mais comum é treinar PINNs com um esquema híbrido: inicialmente com um otimizador de primeira ordem (como Adam), seguido por um método quasi-Newtoniano (como L-BFGS) para refinamento. É justamente o que faremos a seguir. 

# Referências

[1]**LAGARIS, I. E.; LIKAS, A.; FOTIADIS, D. I.** *Artificial neural networks for solving ordinary and partial differential equations*. *IEEE Transactions on Neural Networks*, v. 9, n. 5, p. 987–1000, set. 1998. DOI: https://doi.org/10.1109/72.712178.

[2]__RAISSI, M.; PERDIKARIS, P.; KARNIADAKIS, G. E.__ _Physics-informed neural networks: a deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations. Journal of Computational Physics_, v. 378, p. 686–707, 2019. DOI: https://doi.org/10.1016/j.jcp.2018.10.045

[3]**LIU, D. C.; NOCEDAL, J.** On the limited memory BFGS method for large scale optimization. *Mathematical Programming*, v. 45, p. 503–528, 1989. DOI: https://doi.org/10.1007/BF01589116.


